# FedKG-Maint — MASTER Integrated Closure v1

**One entry point; no retraining and no new architecture experiments.** This notebook binds the **completed Azure Unified v3 Phases 1–4** to the **frozen B1–B4 research findings**, verifies **B4 selected FedAvg MLP out-of-factory scores** against their original labels and metadata, and prepares **strictly historical local-RDF knowledge-graph evidence and paired LLM inputs** for the unfinished Phase 5.

### Fixed research decision
- Selected candidate: **25→32→8→1 MLP, FedAvg, 25 full features**, based on **B4 pilot**, seed 42, 8 rounds, three training factories per LOFO fold.
- Four-fold development mean PR-AUC: **0.762308**, worst fold: **0.742111**. Actionable events: **40/46** with false-alarm frequency about **0.009794 / machine-day** under cross-fold thresholds. These are development/pilot estimates, not blind external results.
- HGB remains the centralized benchmark; LSTM remains an archived comparator.
- This notebook does **not** combine B4 MLP scores with earlier v3 model weights, because **B4 saved prediction arrays, not a final four-factory MLP checkpoint**.
- Original v3 Differential Privacy is a **separate, conditional record-level DP experiment** and DOES NOT protect this selected B4 MLP.
- No fresh training, no NASA dataset, and no LLM API calls by default.

**Run all cells in Colab on your existing Drive.** Input runs remain read-only; all newly generated files go to `unified_runs/fedkg_master_integration_v1/`. Re-running this notebook safely overwrites its own derived integration reports, never the v3/B4 sources.

In [ ]:
from pathlib import Path
import os, json, hashlib, re, sys, subprocess
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
from IPython.display import display
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
ROOT = Path(os.environ.get('FEDKG_DRIVE_ROOT','/content/drive/MyDrive/FedKG-Maint')).expanduser().resolve()
RUNS=ROOT/'unified_runs'
V3=RUNS/'unified_v3_final_20261025'
B=RUNS/'phaseB_development_v1'
B4=RUNS/'phaseB4_generalization_pilot_v1'
OUT=RUNS/'fedkg_master_integration_v1'
assert V3.exists() and (V3/'RUN_COMPLETE.json').exists(), 'Missing completed unified v3 artifacts.'
assert (B4/'B4_COMPLETE.json').exists(), 'Run/finish the existing B4 pilot. No retraining is performed by this master.'
assert (B4/'results'/'B4_model_fold_metrics.csv').exists()
assert (B4/'results'/'B4_operational_summary.csv').exists()
assert (B4/'experiment_lock.json').exists()
assert OUT.resolve() not in [V3.resolve(),B.resolve(),B4.resolve()]
for name in ('reports','results','phase5_inputs'):(OUT/name).mkdir(parents=True,exist_ok=True)
def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as fd:
        for chunk in iter(lambda:fd.read(1<<20),b''):h.update(chunk)
    return h.hexdigest()
SOURCE_PATHS={
    'v3_validation':V3/'prep_v3'/'processed'/'validation.parquet',
    'v3_manifest':V3/'prep_v3'/'manifests'/'factory_manifest.csv',
    'v3_features':V3/'prep_v3'/'configs'/'feature_groups.json',
    'b4_model_metrics':B4/'results'/'B4_model_fold_metrics.csv',
    'b4_operational':B4/'results'/'B4_operational_summary.csv',
    'b4_lock':B4/'experiment_lock.json',
}
for name,p in SOURCE_PATHS.items():assert p.exists(),f'Missing {name}: {p}'
HASHES={name:sha256(p) for name,p in SOURCE_PATHS.items()}
print('Inputs verified | Unified v3, B4 pilot')
print('Working output:',OUT)

## 1 · Load the frozen research findings (without training)

HGB was the strongest **centralized** comparison, but this architecture decision relies on the **cross-factory federated comparison**, robustness in factory 4, and operational errors. Source tables are read from previous completed experiments. These are not interchangeable with NASA RUL or other evaluation protocols.

In [ ]:
FOLDS=['factory_1','factory_2','factory_3','factory_4']
FEATURES=json.loads(SOURCE_PATHS['v3_features'].read_text())['feature_sets']['full']
assert len(FEATURES)==25
LOCK=json.loads(SOURCE_PATHS['b4_lock'].read_text())
assert LOCK['mode']=='pilot' and LOCK['seeds']==[42] and LOCK['rounds']==8
assert LOCK['cap_per_training_client']==6000
assert LOCK['architectures']==['MLP','LSTM']
M=pd.read_csv(SOURCE_PATHS['b4_model_metrics'])
OP=pd.read_csv(SOURCE_PATHS['b4_operational'])
assert len(M)==24 and set(M.fold)==set(FOLDS)
assert not M.duplicated(['fold','architecture','variant','seed']).any()
assert set(M.seed)=={42} and set(M.rounds)=={8}
primary=M[(M.architecture=='MLP')&(M.variant=='full')&(M.seed==42)].copy()
assert len(primary)==4
assert np.isclose(primary.pr_auc.mean(),0.762308,atol=1e-5)
assert np.isclose(primary.pr_auc.min(),0.742111,atol=1e-5)
ops=OP[(OP.architecture=='MLP')&(OP.variant=='full')].iloc[0]
assert int(ops.total_detected_events)==40 and int(ops.total_eligible_events)==46
assert int(ops.total_false_alerts)==27
assert np.isclose(ops.pooled_far,.009794,atol=1e-5)
B4_SUMMARY=pd.read_csv(B4/'results'/'B4_model_summary.csv')
print('Frozen candidate: MLP / FedAvg / full / B4 pilot')
display(B4_SUMMARY.sort_values('robust_score',ascending=False))
print('Selected per-factory scores:')
display(primary[['fold','seed','pr_auc','roc_auc','prevalence']])
print('Operational summary:')
display(OP[['variant','architecture','pooled_event_recall','pooled_far']])
if (B/'results'/'B1_hgb_lofo_summary.csv').exists():
    print('Archived B1 HGB LOFO ablation:')
    display(pd.read_csv(B/'results'/'B1_hgb_lofo_summary.csv'))
else:print('B1 summary is optional for integration; original B1 result not found on Drive.')

## 2 · Reconnect selected B4 predictions to real v3 observation keys

B4's saved arrays are **fold-specific out-of-factory scores**. We reconstruct the exact 24-hour window endpoints and verify (1) machine/time row hash saved during B4, (2) array lengths, (3) each fold's reported PR-AUC, (4) source invariants. The resulting 66,160-row table is a **development-only score index** for KG grounding, **not** a newly trained or private model.

In [ ]:
VA=pd.read_parquet(SOURCE_PATHS['v3_validation'])
VA['datetime']=pd.to_datetime(VA['datetime'])
VA=VA.sort_values(['machineID','datetime']).reset_index(drop=True)
assert set(VA.client_id)==set(FOLDS)
assert not VA[['machineID','datetime']].duplicated().any()
assert set(['machineID','datetime','client_id','label_failure_24h']+FEATURES)<=set(VA.columns)
assert VA[FEATURES].notna().all().all()
FACTORY_MANIFEST=pd.read_csv(SOURCE_PATHS['v3_manifest'])
assert FACTORY_MANIFEST.machineID.is_unique
assert set(VA.machineID).isdisjoint(set(FACTORY_MANIFEST.loc[FACTORY_MANIFEST.client_id.eq('factory_5'),'machineID']))
WINDOW=24
endpoints=[]
for mid,g in VA.groupby('machineID',sort=False):
    assert len(g)>=WINDOW
    assert g.index[-1]-g.index[0]+1==len(g)
    assert (g.datetime.diff().dropna()==pd.Timedelta(hours=1)).all()
    endpoints.extend(g.index.to_numpy()[WINDOW-1:])
endpoints=np.array(endpoints,dtype=np.int64)
reconstructed=[]
for f in FOLDS:
    ix=endpoints[VA.iloc[endpoints].client_id.eq(f).to_numpy()]
    meta=VA.iloc[ix][['machineID','datetime']].reset_index(drop=True)
    digest=hashlib.sha256(meta.to_csv(index=False).encode()).hexdigest()
    stored=B4/'predictions'/f'{f}__MLP__full__seed42.npz'
    stored_meta=B4/'predictions'/f'{f}__MLP__full__seed42.json'
    assert stored.exists() and stored_meta.exists(),f'Missing B4 scored development fold: {f}'
    pm=json.loads(stored_meta.read_text())
    assert pm['validation_rows_sha256']==digest
    assert pm['experiment_lock_sha256']==sha256(SOURCE_PATHS['b4_lock'])
    assert pm['architecture']=='MLP' and pm['variant']=='full' and pm['seed']==42
    with np.load(stored) as obj:scores=obj['scores'].astype('float64')
    assert scores.shape==(len(ix),) and np.isfinite(scores).all()
    part=VA.iloc[ix][['machineID','datetime','client_id','label_failure_24h','volt','rotate','pressure','vibration']].copy().reset_index(drop=True)
    part['risk_score']=scores
    part['model_id']='B4_FedAvg_MLP_full_seed42_LOFO_'+f
    row=primary.loc[primary.fold.eq(f)].iloc[0]
    assert len(part)==int(row.validation_rows)
    observed_ap=average_precision_score(part.label_failure_24h,scores)
    assert np.isclose(observed_ap,float(row.pr_auc),atol=2e-6),f'{f} AP mismatch'
    reconstructed.append(part)
SCORES=pd.concat(reconstructed,ignore_index=True)
assert len(SCORES)==66160
assert not SCORES.duplicated(['machineID','datetime','client_id']).any()
score_path=OUT/'results'/'selected_B4_dev_predictions_internal.csv'
SCORES.to_csv(score_path,index=False)
print('PASS: all 4 machine/timestamp SHA256 alignments + AP reproductions')
print('Development score rows:',len(SCORES))
print('Internal-only scored metadata:',score_path)

## 3 · Choose 12 **development-only** Phase 5 examples

Each of factories 1–4 contributes one low-, medium-, and high-risk case (12 total). Sampling is based on risk rankings, **never on true future failure labels**; labels are not exported to prompts. All conditions receive the identical current sensor state and risk score. This staging experiment is **not a held-out factory-5 explanation result**.

In [ ]:
from collections import defaultdict
CASES=[]
for factory in FOLDS:
    pool=SCORES[SCORES.client_id.eq(factory)].copy().sort_values('risk_score').reset_index(drop=True)
    assert len(pool)>100
    used=set()
    for bucket,pct in [('low',.02),('mid',.50),('high',.98)]:
        rank=int(round((len(pool)-1)*pct))
        order=sorted(range(len(pool)),key=lambda k:(abs(k-rank),k))
        chosen=None
        for idx in order:
            mid=str(pool.iloc[idx].machineID)
            if mid not in used:
                chosen=pool.iloc[idx]; used.add(mid);break
        assert chosen is not None
        CASES.append({'case_id':f'{factory}_{bucket}', 'factory':factory,
                      'machineID':str(int(chosen.machineID)),
                      'prediction_timestamp':pd.Timestamp(chosen.datetime).isoformat(),
                      'risk_score':float(chosen.risk_score), 'risk_stratum':bucket,
                      'model_id':str(chosen.model_id),
                      'current_sensors':{s:float(chosen[s]) for s in ('volt','rotate','pressure','vibration')}})
assert len(CASES)==12
assert len({x['case_id'] for x in CASES})==12
print('12 cases locked (one low/mid/high per factory), no test labels exposed:')
display(pd.DataFrame([{k:v for k,v in p.items() if k!='current_sensors'} for p in CASES]))

## 4 · Ground 12 cases in the existing RDF Knowledge Graphs

Read the *existing* Phase 4 RDF Turtle files, one factory at a time. A machine's graph is never queried from a different factory. Observed error, maintenance, and failure events must have **source timestamps strictly earlier** than the prediction; full-day telemetry summaries must belong to **completed previous days**. Evidence is descriptive and cannot prove the cause of a future failure.

**No LLM calls yet.** Evidence packs and matched with/without-KG prompts are generated for the next phase.

In [ ]:
try:
    from rdflib import Graph,Namespace,RDF,RDFS
except ImportError:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','rdflib'])
    from rdflib import Graph,Namespace,RDF,RDFS
from urllib.parse import quote
ONT=Namespace('https://example.org/fedkg-maint/ontology#')
RES=Namespace('https://example.org/fedkg-maint/resource/')
def normalize_mid(mid):return str(int(float(mid)))
def machine_uri(factory,mid):
    return RES[f'{quote(str(factory),safe="")}/machine/{quote(normalize_mid(mid),safe="")}']
def graph_facts(graph,factory,mid,as_of,max_per_type=6,max_days=7):
    t0=pd.Timestamp(as_of)
    node=machine_uri(factory,mid)
    assert (node,RDF.type,ONT.Machine) in graph,f'Machine {mid} not in {factory} KG'
    facts=[]
    for edge,kind,refedge in [(ONT.hasErrorOccurrence,'error',ONT.hasErrorCode),
                              (ONT.hasMaintenanceEvent,'maintenance',ONT.targetsComponent),
                              (ONT.hasFailureEvent,'failure',ONT.failureOfComponent)]:
        entries=[]
        for event in graph.objects(node,edge):
            timestamp=graph.value(event,ONT.occurredAt)
            if timestamp is None:continue
            ts=pd.Timestamp(str(timestamp))
            if not (t0-pd.Timedelta(days=30)<=ts<t0):continue
            ref=graph.value(event,refedge)
            if kind=='error': val=graph.value(ref,ONT.errorCode) or graph.value(ref,RDFS.label)
            else:val=graph.value(ref,ONT.componentName) or graph.value(ref,RDFS.label)
            entries.append((ts,str(val or 'unknown')))
        for ts,val in sorted(entries,reverse=True)[:max_per_type]:
            facts.append({'source_type':kind,'timestamp':ts.isoformat(),
                          'fact':f'Recorded {kind}: {val} at {ts.isoformat()}; association is not causation.'})
    days=[]
    for item in graph.objects(node,ONT.hasDailyTelemetry):
        d=graph.value(item,ONT.dailyDate)
        if d is None:continue
        day=pd.Timestamp(str(d)).normalize()
        if not (t0.normalize()-pd.Timedelta(days=max_days)<=day<t0.normalize()):continue
        means={sensor:graph.value(item,ONT[f'{sensor}Mean']) for sensor in ('volt','rotate','pressure','vibration')}
        if any(x is None for x in means.values()):continue
        days.append((day,{k:float(v) for k,v in means.items()}))
    for day,means in sorted(days,reverse=True)[:max_days]:
        facts.append({'source_type':'completed_day_telemetry','timestamp':day.isoformat(),
                      'fact':f'Completed day {day.date()} sensor means: '+', '.join(f'{s}={v:.3f}' for s,v in means.items())})
    facts.sort(key=lambda x:x['timestamp'],reverse=True)
    for j,item in enumerate(facts,1):item['evidence_id']=f'KG-{j:03d}'
    # Time guards include complete previous-day rule.
    for item in facts:
        ts=pd.Timestamp(item['timestamp'])
        assert ts<t0,(factory,mid,item)
        if item['source_type']=='completed_day_telemetry':assert ts.normalize()<t0.normalize()
    return facts

GRAPH_ROOT=V3/'phase4_local_knowledge_graphs_v1'/'graphs'
assert (V3/'phase4_local_knowledge_graphs_v1'/'ontology'/'fedkg_shared_ontology.ttl').exists()
PACKS=[]
for factory in FOLDS:
    path=GRAPH_ROOT/f'{factory}.ttl'
    assert path.exists(),f'Missing local RDF graph: {path}'
    g=Graph();g.parse(str(path),format='turtle')
    for case in [c for c in CASES if c['factory']==factory]:
        facts=graph_facts(g,factory,case['machineID'],case['prediction_timestamp'])
        PACKS.append({**case,'kg_facts':facts,'kg_fact_count':len(facts)})
    print('KG linked:',factory,'triples:',len(g),'cases:',3)
    del g
assert len(PACKS)==12
for case in PACKS:
    ids=[f['evidence_id'] for f in case['kg_facts']]
    assert len(ids)==len(set(ids))
    assert all(pd.Timestamp(f['timestamp'])<pd.Timestamp(case['prediction_timestamp']) for f in case['kg_facts'])
packfile=OUT/'phase5_inputs'/'selected_MLP_KG_evidence_12_cases.json'
packfile.write_text(json.dumps(PACKS,ensure_ascii=False,indent=2),encoding='utf-8')
print('PASS: 12 score→RDF evidence joins with strict as-of guards.')
print('Saved:',packfile)

## 5 · Prepare a controlled WITH vs WITHOUT KG ablation (no API calls)

The same factual base context is sent in both conditions. **Only the KG evidence list differs**. Never disclose the offline future-failure labels or an artificial `condition` flag to the LLM. Do not interpret risk scores as calibrated probabilities. Evidence citation IDs are not proof of semantic correctness. If later using a third-party LLM, first consider whether the inputs contain restricted factory information.

In [ ]:
SYSTEM_TEXT=(
  'You are an industrial predictive-maintenance assistant. Use ONLY supplied information. '
  'Explain a MODEL RISK SCORE (not a calibrated probability), distinguish observations from hypotheses, '
  'never claim a failure happened unless there is historical evidence, and give low-risk inspection steps. '
  'Every factual claim should cite supplied evidence IDs, or state insufficient evidence. '
  'Never invent measurements, maintenance history, causal mechanisms, or a failure diagnosis. '
  'Return JSON with keys summary, claims, recommended_actions, uncertainty.'
)
def make_prompt(case,use_kg):
    base={
       'case_id':case['case_id'], 'machineID':case['machineID'], 'factory':case['factory'],
       'prediction_timestamp':case['prediction_timestamp'],
       'risk_score_not_calibrated_probability':case['risk_score'],
       'current_sensor_observations':case['current_sensors'],
       'instruction':'Only use facts in available_facts. No future outcomes or target labels are known.',
       'available_facts':case['kg_facts'] if use_kg else [],
    }
    return [{'role':'system','content':SYSTEM_TEXT},{'role':'user','content':json.dumps(base,ensure_ascii=False)}]
ROWS=[]
for case in PACKS:
    for use_kg in (False,True):
        ROWS.append({'case_id':case['case_id'],
                     'condition':'with_kg' if use_kg else 'without_kg',
                     'messages':make_prompt(case,use_kg)})
assert len(ROWS)==24
# Demonstrate condition parity except for the supplied KG facts.
for case in PACKS:
    a=json.loads(make_prompt(case,False)[1]['content'])
    b=json.loads(make_prompt(case,True)[1]['content'])
    af=a.pop('available_facts');bf=b.pop('available_facts')
    assert af==[] and a==b
    assert bf==case['kg_facts']
pairedfile=OUT/'phase5_inputs'/'matched_24_prompts_internal.jsonl'
pairedfile.write_text(''.join(json.dumps(row,ensure_ascii=False)+'\n' for row in ROWS),encoding='utf-8')
review=pd.DataFrame([{
    'case_id':r['case_id'],'condition':r['condition'],
    'faithfulness_1to5':'','evidence_support_1to5':'','actionability_1to5':'',
    'unsupported_claim_count':'','reviewer_notes':''} for r in ROWS])
review.to_csv(OUT/'phase5_inputs'/'human_evaluation_blank.csv',index=False)
print('Prepared 12 paired cases, 24 prompt inputs; LLM calls made: ZERO')
print('Saved:',pairedfile)

## 6 · Final merged closure report and remaining milestones

**Integration closure is not full project closure.** We have a frozen, pilot-selected prediction architecture and a functioning data→prediction→local-KG→paired-prompt chain **for development factories**. The remaining course-scope work is (a) **actual LLM ablation and human evaluation**, (b) **NASA C-MAPSS RUL** if required by supervisor, and (c) **one final all-development-factory MLP checkpoint / fresh privacy evaluation** if deployment or B4-model DP is required. Those are distinct requirements, not more architecture search. This integration does not falsely claim them finished.

In [ ]:
assert all(sha256(p)==HASHES[k] for k,p in SOURCE_PATHS.items()),'Source artifact changed during integration'
CLOSURE={
    'project':'FedKG-Maint',
    'integrated_sources':['Unified v3 Phases1–4','B1–B3 historical experiments','B4 pilot selected MLP'],
    'prediction_architecture_decision':'FedAvg MLP 25→32→8→1, full 25-feature inputs',
    'selection_protocol':'4 development-factory LOFO folds; B4 pilot, 8 rounds, seed42',
    'mean_pr_auc_development':float(primary.pr_auc.mean()),
    'worst_pr_auc_development':float(primary.pr_auc.min()),
    'event_recall_development':int(ops.total_detected_events)/int(ops.total_eligible_events),
    'false_alarms_per_machine_day_development':float(ops.pooled_far),
    'predictions_reconnected_and_verified':len(SCORES),
    'kg_paired_cases_built':len(PACKS),
    'paired_prompts_prepared':len(ROWS),
    'source_sha256':HASHES,
    'llm_calls_executed':0,
    'privacy_B4_selected_model':'NONE; v3 conditional DP does not transfer',
    'single_B4_final_deployable_checkpoint_available':False,
    'nasa_rul_implemented':False,
    'heldout_factory5_new_blind_evaluation':False,
    'limits':[
      'B4 single-seed pilot, exploratory development LOFO, no independent external factory',
      'Histories from synthetic Azure data may introduce operationally unrealistic predictive signals',
      'B4 stores scored folds, not a deployable four-client trained checkpoint',
      'No end-to-end DP or cryptographic Secure Aggregation is demonstrated',
      'LLM explanation quality not tested; human evaluation sheet is blank',
    ]
}
(OUT/'reports'/'MASTER_CLOSURE.json').write_text(json.dumps(CLOSURE,indent=2,ensure_ascii=False))
REPORT=['# FedKG-Maint — Master Research Integration Closure','',
        '**STATUS: Research artifacts integrated; unfinished phases remain transparently open.**','',
        '## Frozen model choice','**FedAvg MLP 25→32→8→1**, full 25 features, B4 pilot.','',
        f'- Mean development LOFO PR-AUC: **{CLOSURE["mean_pr_auc_development"]:.4f}**',
        f'- Worst development fold: **{CLOSURE["worst_pr_auc_development"]:.4f}**',
        f'- Actionable event recall: **{ops.total_detected_events}/{ops.total_eligible_events}**',
        f'- False alarms per machine-day: **{ops.pooled_far:.5f}**','',
        '## Verified integration',f'- B4 prediction rows verified: **{len(SCORES):,}**',
        f'- Local KG cases prepared: **{len(PACKS)}**',f'- Balanced LLM prompts: **{len(ROWS)}** (not executed)','',
        '## Important limitations', *[f'- {x}' for x in CLOSURE['limits']], '',
        '## Remaining required work','- Execute and review Phase5 LLM pairwise experiment and human evaluation.',
        '- Complete NASA C-MAPSS RUL benchmark if still required by project specification.',
        '- A final B4 MLP global checkpoint is NOT in prior output; if an operational model or matched DP comparison is needed, one explicitly controlled final fit will be required. No new architecture search.','',
        '## Reproducibility','- Source files were read-only and verified by SHA256.',
        '- Archived original notebooks remain separate as provenance, not primary notebooks.',
        '- Master output directory is not a publication-safe DP release.']
(OUT/'reports'/'MASTER_CLOSURE.md').write_text('\n'.join(REPORT))
print('MERGE COMPLETE — NO MODEL TRAINING, NO LLM CALLS, NO FACTORY 5 EVALUATION')
print('Master report:',OUT/'reports'/'MASTER_CLOSURE.md')
display(pd.DataFrame([{'component':k,'status':v} for k,v in [
    ('B4 MLP score alignment','Verified'),('Development prediction→KG integration','Prepared and validated'),
    ('Paired Phase5 LLM input set','Prepared, 0 calls'),('Differential privacy of B4 weights','NOT established'),
    ('NASA RUL','Not run')]]))